# 06.2 — Interactive UMAP Visualization

Uses the reusable visualization pipeline (`speciesOT/visualization/umap_pipeline.py`) 
to compute UMAP and generate interactive Plotly HTML files.

**Workflow:**
1. Compute reference UMAP + project predictions + marker genes (saved to .npz)
2. Generate interactive HTML plots from the .npz

**Environment:** Run in the `analysis` conda env (scanpy 1.12, umap-learn 0.5.11, plotly).

**Output HTML files:**
- `interactive_all_celltypes.html` — toggle individual cell types on/off
- `interactive_species_overlay.html` — toggle mouse/human
- `interactive_marker_genes.html` — dropdown to select gene expression overlay
- `interactive_impact_or.html` — IMPACT-OR predictions with hover metadata
- `interactive_cellot.html` — CellOT predictions with hover metadata

In [1]:
import sys
sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")

from pathlib import Path
from visualization.umap_pipeline import compute_and_save_umap, generate_interactive_plots

RESULTS = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/results/speciesot_cd8")
ANALYSIS_DIR = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis")

DATA_PATH = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_holdout_v07.h5ad")
if not DATA_PATH.exists():
    DATA_PATH = RESULTS / "data.h5ad"

PREDICTIONS = {
    "IMPACT-OR": RESULTS / "impact_or" / "evals_ood_data_space" / "imputed.h5ad",
    "CellOT": RESULTS / "cellot" / "evals_ood_data_space" / "imputed.h5ad",
}

NPZ_PATH = ANALYSIS_DIR / "umap_reference_cd8.npz"

print(f"Data: {DATA_PATH}")
print(f"Predictions: {list(PREDICTIONS.keys())}")
print("Setup complete.")

Data: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_holdout_v07.h5ad
Predictions: ['IMPACT-OR', 'CellOT']
Setup complete.


## Step 1: Compute Reference UMAP

PCA (50 comps) → Neighbors (k=15) → UMAP (min_dist=0.3, seed=42).
Projects model predictions via sc.tl.ingest. Computes marker genes.
Saves everything to a .npz file for reproducible visualization.

In [2]:
compute_and_save_umap(
    data_path=DATA_PATH,
    output_path=NPZ_PATH,
    predictions=PREDICTIONS,
    n_comps=50,
    n_neighbors=15,
    min_dist=0.3,
    random_state=42,
)

Loading data from /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_holdout_v07.h5ad ...
Computing PCA (n_comps=50) ...
Computing neighbors (n_neighbors=15, n_pcs=50) ...


/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Computing UMAP (min_dist=0.3, random_state=42) ...
UMAP done: (12836, 2)
Computing marker genes (groupby=cell_type) ...


/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:457: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:459: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/scanpy/tools/_ran

  Projecting IMPACT-OR (195 cells) via sc.tl.ingest ...
  IMPACT-OR: projected
  Projecting CellOT (6223 cells) via sc.tl.ingest ...
  CellOT: projected

Saved to /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/umap_reference_cd8.npz (3.8 MB)


## Step 2: Generate Interactive Plots

Loads the .npz and produces Plotly HTML files. Open them in a browser
to hover over cells, toggle cell types, and explore marker gene expression.

In [3]:
generate_interactive_plots(
    npz_path=NPZ_PATH,
    output_dir=ANALYSIS_DIR,
    holdout_ct_id="CL:0000625",
)

Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_all_celltypes.html
Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_species_overlay.html
Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_marker_genes.html
Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_impact_or.html
Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_cellot.html

All interactive plots saved to /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/


## Next Steps

Open the HTML files in a browser:

1. **interactive_all_celltypes.html** — Click legend entries to show/hide individual cell types.
   Use this to isolate CD8+ T cells and see where they cluster relative to thymocytes.

2. **interactive_species_overlay.html** — Toggle mouse vs human to check mixing quality.

3. **interactive_marker_genes.html** — Use the dropdown to overlay marker gene expression.
   Check if outlier CD8 cells express the expected markers.

4. **interactive_impact_or.html** / **interactive_cellot.html** — Hover over green (actual CD8) 
   and orange (predicted) dots to inspect donor, tissue, and cell type metadata.
   Look for blue dots scattered near pulmonary alveolar clusters.